In [0]:
df = spark\
.read\
.option("inferSchema", "true")\
.option("header", "true")\
.csv("/databricks-datasets/flights/departuredelays.csv")

In [0]:
df.createOrReplaceTempView("flight_df")

In [0]:
%sql
select count(*) from flight_df

In [0]:
df.printSchema()

In [0]:
df.count()

In [0]:
%sql
select * from flight_df limit 10

In [0]:
%sql
select count(1),origin from flight_df group by origin order by 1 desc

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
select max(date),min(date) from flight_df

In [0]:
%sql
select count(1), date  from flight_df group by date having count(1)> 1 order by 1 desc limit 100

In [0]:
%sql
select * from flight_df where date=3180600 limit 100

In [0]:
%sql
select count(1),date,origin,destination from flight_df group by date,origin,destination having count(1) > 1 limit 100

In [0]:
%sql
select * from flight_df where date='1120815' and origin='DEN' and destination='PDX'

In [0]:
%sql
select a.*,
row_number() over(partition by origin order by date) as flight_rank,
dense_rank() over(partition by 1 order by date) date_rank
 from flight_df a where a.date='1120815' and a.origin='DEN' and a.destination='PDX' 

In [0]:
%sql
select a.*,
row_number() over(partition by origin order by date) as flight_rank,
dense_rank() over(partition by 1 order by date) date_rank
 from flight_df a  order by date limit 100

In [0]:
%sql
select a.*,
row_number() over(partition by origin order by date) as flight_rank,
date_add(to_timestamp('1999-12-31', 'yyyy-MM-dd'),dense_rank() over(partition by 1 order by date)) Date
 from flight_df a  order by date limit 10

In [0]:
from pyspark.sql.functions import round

In [0]:
%sql
Create or replace temp view flight_Data_crafted as 
select a.*,
row_number() over(partition by origin order by date) as flight_rank,
dense_rank() over(partition by 1 order by date) Day_rank,
date_add(to_timestamp('1999-12-31', 'yyyy-MM-dd'),int(round(dense_rank() over(partition by 1 order by date)/24))) new_date
 from flight_df a  order by date 

In [0]:
%sql
select * from flight_Data_crafted order by Day_rank desc limit 10

In [0]:
from pyspark.sql.functions import window,col,column,desc
df_new =spark.sql("select new_date as date,delay,origin,destination from flight_Data_crafted")

In [0]:
df_new.printSchema()

In [0]:
df_7_days =df_new\
    .selectExpr("origin","destination","delay","date")\
    .groupBy(col("origin"),window(col("date"),"7 days"))\
          .sum("delay")
display(df_7_days.take(5))

In [0]:
df_30_days =df_new\
    .selectExpr("origin","destination","delay","date")\
    .groupBy(col("origin"),window(col("date"),"7 days"))\
          .sum("delay")
display(df_30_days.sort("origin").take(5)

In [0]:
df_30_days =df_new\
    .selectExpr("origin","destination","delay","date")\
    .groupBy(col("origin"),window(col("date"),"7 days"))\
          .sum("delay")
display(df_30_days.where("origin='LAX'").take(5))

In [0]:
display(df_new.take(5))

In [0]:
%sql
select * from flight_Data_crafted limit 10